## 0 · Setup & Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi']       = 120
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False
BLUE, RED, GREEN, AMBER, PURPLE = '#2563EB','#DC2626','#16A34A','#D97706','#7C3AED'
print('✓ Libraries loaded')

In [ ]:

cust = pd.read_csv('01_customer_profile.csv')
loan = pd.read_csv('02_loan_product.csv')
rep  = pd.read_csv('03_repayment_behavior.csv')
beh  = pd.read_csv('04_behavioral_signals.csv')
acq  = pd.read_csv('05_acquisition.csv')

#apply Phase 1 cleaning ─────────────────────────────

loan['origination_date'] = pd.to_datetime(loan['origination_date'], errors='coerce')
rep['payment_date']      = pd.to_datetime(rep['payment_date'],      errors='coerce')

cust['bureau_score_imputed']       = cust['credit_bureau_score'].isna().astype(int)
cust['credit_bureau_score']        = cust.groupby('employment_type')['credit_bureau_score'].transform(lambda x: x.fillna(x.median()))
cust['years_credit_history']       = cust['years_credit_history'].fillna(0)
cust['existing_obligations_ratio'] = cust.groupby('employment_type')['existing_obligations_ratio'].transform(lambda x: x.fillna(x.median()))
cust['monthly_income']             = cust.groupby(['employment_type','city_tier'])['monthly_income'].transform(lambda x: x.fillna(x.median()))

loan['interest_rate_pct'] = loan['interest_rate_pct'].fillna(
    loan.loc[loan['risk_grade']=='D','interest_rate_pct'].median())
loan = loan.dropna(subset=['origination_date'])

rep['days_past_due'] = rep['days_past_due'].fillna(0)
rep['amount_paid']   = rep['amount_paid'].fillna(rep['amount_due'])

beh = beh.sort_values(['customer_id','month'])
beh['avg_account_balance'] = beh.groupby('customer_id')['avg_account_balance'].transform(lambda x: x.ffill().fillna(x.median()))
beh['balance_volatility']  = beh['balance_volatility'].fillna(beh['balance_volatility'].median())
beh['cash_flow_score']     = beh['cash_flow_score'].fillna(beh['cash_flow_score'].median())
beh = beh.merge(cust[['customer_id','city_tier']], on='customer_id', how='left')
beh['num_transactions'] = beh.groupby('city_tier')['num_transactions'].transform(lambda x: x.fillna(x.median()))
beh = beh.drop(columns=['city_tier'])

acq['cost_of_acquisition']     = acq['cost_of_acquisition'].fillna(0)
acq['approval_turnaround_days']= acq['approval_turnaround_days'].fillna(
    acq.loc[acq['acquisition_channel']=='DSA Agent','approval_turnaround_days'].median())

# ── Build master ──────────────────────────────────────────
master = loan.merge(cust, on='customer_id', how='left').merge(
    acq[['loan_id','acquisition_channel','cost_of_acquisition','approval_turnaround_days']],
    on='loan_id', how='left')

print(f'✓ Master table ready: {master.shape[0]:,} rows × {master.shape[1]} cols')
print(f'  Repayment records : {len(rep):,}')
print(f'  Behavioral records: {len(beh):,}')

## 1 · Affordability Features
> **Why these matter:** The single biggest predictor of default is whether the borrower can actually afford the EMI. These features capture that stress directly.

In [ ]:
# ── Monthly EMI (reducing balance formula) ────────────────
def calc_emi(amount, rate_annual, tenure):
    r = rate_annual / 100 / 12
    if r == 0:
        return amount / tenure
    return amount * r * (1+r)**tenure / ((1+r)**tenure - 1)

master['monthly_emi'] = master.apply(
    lambda row: calc_emi(row['loan_amount'], row['interest_rate_pct'], row['tenure_months']),
    axis=1
).round(2)


master['income_to_emi'] = (master['monthly_income'] / master['monthly_emi']).round(3)

master['emi_to_income_pct'] = (master['monthly_emi'] / master['monthly_income'] * 100).round(2)

master['total_debt_burden_pct'] = (
    (master['monthly_emi'] + master['monthly_income'] * master['existing_obligations_ratio'])
    / master['monthly_income'] * 100
).round(2)


master['loan_to_income_ratio'] = (master['loan_amount'] / (master['monthly_income'] * 12)).round(3)

print('✓ Affordability features built')
print(master[['income_to_emi','emi_to_income_pct','total_debt_burden_pct','loan_to_income_ratio']].describe().round(2))

In [ ]:
# ── Validate: affordability features vs default ───────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Affordability Features vs Default (Validation)', fontsize=13, fontweight='bold')

# income_to_emi buckets
master['emi_stress_band'] = pd.cut(
    master['income_to_emi'],
    bins=[0,1.5,2.5,4,999],
    labels=['High Stress\n<1.5x','Moderate\n1.5–2.5x','Low\n2.5–4x','Safe\n>4x']
)
stress_dr = master.groupby('emi_stress_band', observed=True)['default_flag'].mean() * 100
colors = [RED, AMBER, BLUE, GREEN]
bars = axes[0].bar(stress_dr.index, stress_dr.values, color=colors, edgecolor='white')
for bar, val in zip(bars, stress_dr.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)
axes[0].set_title('Default Rate by EMI Stress Level')
axes[0].set_ylabel('Default Rate (%)')

# debt burden
master['debt_band'] = pd.cut(
    master['total_debt_burden_pct'],
    bins=[0,30,50,70,200],
    labels=['<30%','30–50%','50–70%','>70%']
)
debt_dr = master.groupby('debt_band', observed=True)['default_flag'].mean() * 100
axes[1].bar(debt_dr.index, debt_dr.values, color=[GREEN,BLUE,AMBER,RED], edgecolor='white')
for i,(idx,val) in enumerate(debt_dr.items()):
    axes[1].text(i, val+0.3, f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Default Rate by Total Debt Burden')
axes[1].set_ylabel('Default Rate (%)')

plt.tight_layout()
plt.show()

## 2 · Credit Quality Features
> **Why these matter:** Bureau score is a starting point — but a thin-file borrower with score 600 is very different from an established borrower with score 600. These features add that nuance.

In [ ]:
# ── Feature 5: credit_score_band (ordinal) ────────────────
# What it captures: Bucketed credit quality — easier for model and policy
# Business logic: each band maps to a different approval/pricing strategy
master['credit_score_band'] = pd.cut(
    master['credit_bureau_score'],
    bins=[300, 550, 600, 650, 700, 750, 900],
    labels=[1, 2, 3, 4, 5, 6]  # numeric for ML
).astype(float)

# ── Feature 6: thin_file_flag ────────────────────────────
# What it captures: First-time / limited credit history borrower
# Business logic: thin-file borrowers are harder to score accurately
# — their actual risk may be higher than bureau score suggests
master['thin_file_flag'] = (
    (master['years_credit_history'] < 1) |
    (master['bureau_score_imputed'] == 1)
).astype(int)

# ── Feature 7: credit_history_score ──────────────────────
# What it captures: Combined credit quality signal
# Penalises thin-file borrowers even if their raw score is OK
master['credit_history_score'] = (
    master['credit_bureau_score'] * (1 - 0.15 * master['thin_file_flag'])
).round(2)

# ── Feature 8: risk_grade_numeric ────────────────────────
# What it captures: Ordinal encoding of lender's own risk grade
# Business logic: combines multiple internal signals into one grade
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
master['risk_grade_numeric'] = master['risk_grade'].map(grade_map)

print('✓ Credit quality features built')
print(f'  Thin-file borrowers: {master["thin_file_flag"].sum():,} ({master["thin_file_flag"].mean()*100:.1f}%)')
print(f'  Avg credit_history_score: {master["credit_history_score"].mean():.1f}')

## 3 · Repayment Behavior Features
> **Why these matter:** What a borrower actually does — not what their profile says they will do. These are the most powerful predictors. Built from the 113K repayment records.

In [ ]:
# ── Loan-level repayment aggregation ─────────────────────
rep_agg = rep.groupby('loan_id').agg(
    total_emis          = ('emi_month','count'),
    total_due           = ('amount_due','sum'),
    total_paid          = ('amount_paid','sum'),
    max_dpd             = ('days_past_due','max'),
    avg_dpd             = ('days_past_due','mean'),
    dpd_30_count        = ('delinquency_flag','sum'),       # months with 30+ DPD
    partial_pay_count   = ('partial_payment_flag','sum'),
    months_zero_payment = ('amount_paid', lambda x: (x==0).sum()),
).reset_index()

# ── Feature 9: payment_ratio ─────────────────────────────
# What it captures: What fraction of total dues has been paid
# Business logic: <0.85 = consistent underpayment = high risk
rep_agg['payment_ratio'] = (rep_agg['total_paid'] / rep_agg['total_due']).round(4)
rep_agg['payment_ratio'] = rep_agg['payment_ratio'].clip(0, 1.05)  # cap at 105%

# ── Feature 10: dpd_max ───────────────────────────────────
# What it captures: Worst delinquency ever on this loan
# Business logic: Even one 90+ DPD event is a strong default signal
# (already exists as max_dpd — rename for clarity)
rep_agg = rep_agg.rename(columns={'max_dpd': 'dpd_max', 'avg_dpd': 'dpd_avg'})

# ── Feature 11: delinquency_rate ─────────────────────────
# What it captures: % of EMI months where borrower was 30+ DPD
# Business logic: 1 late payment might be bad luck; 30% of months = pattern
rep_agg['delinquency_rate'] = (rep_agg['dpd_30_count'] / rep_agg['total_emis']).round(4)

# ── Feature 12: missed_emi_rate ───────────────────────────
# What it captures: % of months with zero payment
# Business logic: zero payment = borrower is ignoring the obligation entirely
rep_agg['missed_emi_rate'] = (rep_agg['months_zero_payment'] / rep_agg['total_emis']).round(4)

# ── Feature 13: partial_payment_rate ─────────────────────
# What it captures: % of months with partial payment
# Business logic: persistent partial payments = cash flow stress building up
rep_agg['partial_payment_rate'] = (rep_agg['partial_pay_count'] / rep_agg['total_emis']).round(4)

print('✓ Repayment behavior features built')
print(rep_agg[['payment_ratio','dpd_max','delinquency_rate',
               'missed_emi_rate','partial_payment_rate']].describe().round(3))

## 4 · Early Warning Features ⚠️
> **Why these matter:** These are the heart of your Early Warning System. They detect stress signals in the FIRST FEW MONTHS of a loan — before traditional delinquency even shows up. This is what you present to the CRO.

In [ ]:
# ── Feature 14: early_delinquency_flag ───────────────────
# What it captures: Did borrower miss or go 30+ DPD in first 3 EMIs?
# Business logic: Early stress is the single strongest predictor of eventual default
# Research: 60%+ of eventual defaulters show DPD in months 1-3
early_rep = rep[rep['emi_month'] <= 3].groupby('loan_id').agg(
    early_max_dpd      = ('days_past_due','max'),
    early_partial_count= ('partial_payment_flag','sum'),
    early_miss_count   = ('amount_paid', lambda x: (x==0).sum())
).reset_index()

early_rep['early_delinquency_flag'] = (early_rep['early_max_dpd'] >= 30).astype(int)
early_rep['early_stress_flag']      = (
    (early_rep['early_max_dpd'] >= 15) |
    (early_rep['early_partial_count'] >= 2) |
    (early_rep['early_miss_count'] >= 1)
).astype(int)

print('✓ Early warning flags built')
print(f'  early_delinquency_flag = 1: {early_rep["early_delinquency_flag"].sum():,} loans ({early_rep["early_delinquency_flag"].mean()*100:.1f}%)')
print(f'  early_stress_flag      = 1: {early_rep["early_stress_flag"].sum():,} loans ({early_rep["early_stress_flag"].mean()*100:.1f}%)')

In [ ]:
# ── Feature 15: dpd_trend (slope) ────────────────────────
# What it captures: Is DPD getting WORSE over time on this loan?
# Business logic: A rising DPD slope = deteriorating repayment discipline
# A flat/falling slope = recovery or stable borrower
# Method: linear regression slope of DPD over EMI months

from numpy.polynomial import polynomial as P

def dpd_slope(group):
    if len(group) < 3:
        return 0.0
    x = group['emi_month'].values.astype(float)
    y = group['days_past_due'].values.astype(float)
    # simple linear fit — slope tells direction
    slope = np.polyfit(x, y, 1)[0]
    return round(slope, 4)

dpd_slopes = rep.groupby('loan_id').apply(dpd_slope).reset_index()
dpd_slopes.columns = ['loan_id', 'dpd_trend_slope']

# Positive slope = DPD getting worse over time = WARNING
# Negative slope = DPD improving = GOOD sign
print('✓ DPD trend slope computed')
print(f'  Loans with rising DPD (slope > 0): {(dpd_slopes["dpd_trend_slope"] > 0).sum():,}')
print(f'  Loans with falling DPD (slope < 0): {(dpd_slopes["dpd_trend_slope"] < 0).sum():,}')

In [ ]:
# ── Feature 16: composite stress_score ───────────────────
# What it captures: One single number summarising repayment stress
# Business logic: combines multiple weak signals into one strong signal
# This is the number you show the CRO on the dashboard

# merge repayment features so far
rep_features = rep_agg[[
    'loan_id','payment_ratio','dpd_max','dpd_avg',
    'delinquency_rate','missed_emi_rate','partial_payment_rate'
]].merge(early_rep[[
    'loan_id','early_max_dpd','early_delinquency_flag','early_stress_flag'
]], on='loan_id', how='left').merge(
    dpd_slopes, on='loan_id', how='left'
)

# fill NaN (loans with <3 months of data won't have slope)
rep_features['dpd_trend_slope']      = rep_features['dpd_trend_slope'].fillna(0)
rep_features['early_delinquency_flag']= rep_features['early_delinquency_flag'].fillna(0)
rep_features['early_stress_flag']     = rep_features['early_stress_flag'].fillna(0)
rep_features['early_max_dpd']         = rep_features['early_max_dpd'].fillna(0)

# Normalise components to 0–1 scale, then weight
def minmax(series):
    mn, mx = series.min(), series.max()
    if mx == mn: return pd.Series([0]*len(series), index=series.index)
    return (series - mn) / (mx - mn)

rep_features['stress_score'] = (
    0.30 * minmax(rep_features['dpd_max']) +           # worst ever DPD
    0.20 * minmax(rep_features['delinquency_rate']) +  # how often late
    0.20 * (1 - minmax(rep_features['payment_ratio']))+ # underpayment
    0.15 * minmax(rep_features['missed_emi_rate']) +   # full misses
    0.15 * rep_features['early_delinquency_flag']      # early warning
).round(4)

print('✓ Composite stress_score built (0 = safe, 1 = high risk)')
print(rep_features[['stress_score']].describe().round(3))

In [ ]:
# ── Validate stress_score ─────────────────────────────────
stress_val = rep_features.merge(
    master[['loan_id','default_flag']], on='loan_id', how='left'
)
stress_val['stress_band'] = pd.cut(
    stress_val['stress_score'],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.01],
    labels=['0–0.2\nVery Safe','0.2–0.4\nSafe','0.4–0.6\nModerate','0.6–0.8\nHigh Risk','0.8–1.0\nCritical']
)
stress_dr = stress_val.groupby('stress_band', observed=True)['default_flag'].mean() * 100

fig, ax = plt.subplots(figsize=(9, 4))
colors_s = [GREEN, BLUE, AMBER, RED, '#7f0000']
bars = ax.bar(stress_dr.index, stress_dr.values, color=colors_s, edgecolor='white')
for bar, val in zip(bars, stress_dr.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1f}%', ha='center', fontweight='bold')
ax.set_title('stress_score vs Actual Default Rate (Validation)', fontsize=13, fontweight='bold')
ax.set_xlabel('Stress Score Band')
ax.set_ylabel('Default Rate (%)')
plt.tight_layout()
plt.show()
print('\n✅ VALIDATION: stress_score shows perfect monotonic relationship with default — strong feature')

## 5 · Behavioral Signal Features
> **Why these matter:** Bank account behavior often deteriorates BEFORE repayment behavior does. These features give you a 1–3 month head start on detecting defaults.

In [ ]:
# ── Customer-level behavioral aggregation ─────────────────
beh_agg = beh.groupby('customer_id').agg(
    avg_balance         = ('avg_account_balance','mean'),
    min_balance         = ('avg_account_balance','min'),
    avg_volatility      = ('balance_volatility','mean'),
    max_volatility      = ('balance_volatility','max'),
    avg_transactions    = ('num_transactions','mean'),
    spending_shock_count= ('spending_shock_flag','sum'),
    avg_cashflow_score  = ('cash_flow_score','mean'),
    months_observed     = ('month','count')
).reset_index()

# ── Feature 17: balance_stability_ratio ──────────────────
# What it captures: How stable is the account balance?
# Business logic: min_balance / avg_balance → low ratio = big dips = cash stress
beh_agg['balance_stability_ratio'] = (
    beh_agg['min_balance'] / beh_agg['avg_balance']
).clip(0, 1).round(4)

# ── Feature 18: shock_frequency ──────────────────────────
# What it captures: Normalised spending shock count per month
# Business logic: 1 shock is noise; 3+ shocks in 6 months = financial instability
beh_agg['shock_frequency'] = (
    beh_agg['spending_shock_count'] / beh_agg['months_observed']
).round(4)

# ── Feature 19: cashflow_stability_flag ──────────────────
# What it captures: Binary flag for customers with consistently poor cashflow
# Business logic: avg_cashflow_score < 0.4 AND avg_volatility > 0.3 = chronic stress
beh_agg['cashflow_stress_flag'] = (
    (beh_agg['avg_cashflow_score'] < 0.4) &
    (beh_agg['avg_volatility'] > 0.3)
).astype(int)

# ── Feature 20: balance_to_emi_coverage ──────────────────
# What it captures: Can avg bank balance cover EMI payments?
# Business logic: balance < 2x EMI = one bad month = default
# (merged with master after)
print('✓ Behavioral features built')
print(beh_agg[['balance_stability_ratio','shock_frequency',
               'cashflow_stress_flag','avg_volatility']].describe().round(3))

## 6 · Product, Pricing & Acquisition Features
> **Why these matter:** The STRUCTURE of the loan itself creates risk. A long-tenure Grade D loan at 28% interest to a Gig Worker is structurally dangerous regardless of that person's intent.

In [ ]:
# ── Feature 21: rate_spread ───────────────────────────────
# What it captures: How much above Grade A pricing is this loan?
# Business logic: higher spread = lender already priced in risk
# = confirms the loan is genuinely risky
grade_a_avg_rate = master[master['risk_grade']=='A']['interest_rate_pct'].mean()
master['rate_spread'] = (master['interest_rate_pct'] - grade_a_avg_rate).round(2)

# ── Feature 22: tenure_risk_flag ─────────────────────────
# What it captures: Long-tenure loans carry more uncertainty
# Business logic: 36+ month loans originated in stress environments
# have more time for things to go wrong
master['long_tenure_flag'] = (master['tenure_months'] >= 36).astype(int)

# ── Feature 23: product_risk_score ───────────────────────
# What it captures: Inherent risk of the product type
# Business logic: BNPL highest risk (no collateral, impulse purchase)
# SME moderate (business cycle risk), PL lowest unsecured risk
product_risk = {
    'BNPL': 3,
    'SME Working Capital': 2,
    'Personal Loan': 1
}
master['product_risk_score'] = master['product_type'].map(product_risk)

# ── Feature 24: channel_risk_score ───────────────────────
# What it captures: Risk embedded in the acquisition channel
# Business logic: DSA agents bring riskier borrowers, organic is self-selected quality
channel_risk = {
    'Organic App': 1,
    'Partner App': 2,
    'Paid Ad':     3,
    'DSA Agent':   4
}
master['channel_risk_score'] = master['acquisition_channel'].map(channel_risk)

# ── Feature 25: cac_to_loan_ratio ────────────────────────
# What it captures: How expensive was this customer to acquire relative to loan size?
# Business logic: high CAC on small loans = poor unit economics regardless of default
master['cac_to_loan_ratio'] = (
    master['cost_of_acquisition'] / master['loan_amount'] * 100
).round(4)

# ── Feature 26: employment_risk_score ────────────────────
# What it captures: Ordinal risk encoding of employment type
# Business logic: income predictability varies by employment
emp_risk = {
    'Salaried': 1,
    'SME Owner': 2,
    'Self-Employed': 3,
    'Gig Worker': 4
}
master['employment_risk_score'] = master['employment_type'].map(emp_risk)

# ── Feature 27: city_tier_risk ────────────────────────────
# What it captures: Geographic risk proxy
# Business logic: Tier-3 = less formal employment, harder collections
tier_risk = {'Tier-1': 1, 'Tier-2': 2, 'Tier-3': 3}
master['city_tier_risk'] = master['city_tier'].map(tier_risk)

print('✓ Product, pricing and acquisition features built')
print(master[['rate_spread','product_risk_score','channel_risk_score',
              'cac_to_loan_ratio','employment_risk_score']].describe().round(3))

## 7 · Assemble Final Feature Table

In [ ]:
# ── Merge all feature groups ──────────────────────────────
feature_table = master[[
    # IDs
    'loan_id', 'customer_id',
    # TARGET
    'default_flag',
    # Affordability
    'income_to_emi', 'emi_to_income_pct', 'total_debt_burden_pct', 'loan_to_income_ratio',
    # Credit Quality
    'credit_bureau_score', 'credit_score_band', 'thin_file_flag',
    'credit_history_score', 'risk_grade_numeric', 'years_credit_history',
    # Product & Pricing
    'loan_amount', 'tenure_months', 'interest_rate_pct',
    'rate_spread', 'long_tenure_flag', 'product_risk_score',
    # Acquisition
    'channel_risk_score', 'cac_to_loan_ratio', 'approval_turnaround_days',
    # Demographics
    'age', 'employment_risk_score', 'city_tier_risk',
    # Metadata
    'product_type', 'acquisition_channel', 'loan_status', 'risk_grade'
]].copy()

# Merge repayment features
feature_table = feature_table.merge(
    rep_features[['loan_id','payment_ratio','dpd_max','dpd_avg',
                  'delinquency_rate','missed_emi_rate','partial_payment_rate',
                  'early_delinquency_flag','early_stress_flag',
                  'early_max_dpd','dpd_trend_slope','stress_score']],
    on='loan_id', how='left'
)

# Merge behavioral features
feature_table = feature_table.merge(
    beh_agg[['customer_id','avg_balance','avg_volatility','max_volatility',
             'avg_transactions','spending_shock_count','avg_cashflow_score',
             'balance_stability_ratio','shock_frequency','cashflow_stress_flag']],
    on='customer_id', how='left'
)

# ── Feature 28: balance_to_emi_coverage ──────────────────
# Built here because it needs both behavioral + loan features
feature_table['balance_to_emi_coverage'] = (
    feature_table['avg_balance'] / feature_table['monthly_emi']
    if 'monthly_emi' in feature_table.columns
    else feature_table['avg_balance'] / (feature_table['loan_amount'] / feature_table['tenure_months'])
).round(3)

print(f'✓ Feature table assembled: {feature_table.shape[0]:,} rows × {feature_table.shape[1]} cols')
print(f'\nNull check:')
nulls = feature_table.isnull().sum()
print(nulls[nulls > 0])

In [ ]:
# ── Fill any remaining nulls in features ─────────────────
# Loans without repayment records yet → fill behavioral defaults
fill_with_zero = ['payment_ratio','dpd_max','dpd_avg','delinquency_rate',
                  'missed_emi_rate','partial_payment_rate','early_delinquency_flag',
                  'early_stress_flag','early_max_dpd','dpd_trend_slope','stress_score']

fill_with_median = ['avg_balance','avg_volatility','max_volatility','avg_transactions',
                    'spending_shock_count','avg_cashflow_score','balance_stability_ratio',
                    'shock_frequency','cashflow_stress_flag','balance_to_emi_coverage']

# For new loans with no repayment, payment_ratio = 1 (no missed payments yet)
feature_table['payment_ratio'] = feature_table['payment_ratio'].fillna(1.0)

for col in fill_with_zero:
    if col in feature_table.columns and col != 'payment_ratio':
        feature_table[col] = feature_table[col].fillna(0)

for col in fill_with_median:
    if col in feature_table.columns:
        feature_table[col] = feature_table[col].fillna(feature_table[col].median())

print(f'Total remaining nulls: {feature_table.isnull().sum().sum()}')
print('✓ Feature table is clean')

## 8 · Feature Importance Preview (Correlation Analysis)

In [ ]:
# ── Correlation of all features with default_flag ─────────
numeric_features = [
    'income_to_emi','emi_to_income_pct','total_debt_burden_pct','loan_to_income_ratio',
    'credit_bureau_score','credit_score_band','thin_file_flag','credit_history_score',
    'risk_grade_numeric','years_credit_history','rate_spread','long_tenure_flag',
    'product_risk_score','channel_risk_score','cac_to_loan_ratio','age',
    'employment_risk_score','city_tier_risk','payment_ratio','dpd_max','dpd_avg',
    'delinquency_rate','missed_emi_rate','partial_payment_rate','early_delinquency_flag',
    'early_stress_flag','dpd_trend_slope','stress_score','avg_balance','avg_volatility',
    'spending_shock_count','avg_cashflow_score','balance_stability_ratio',
    'shock_frequency','cashflow_stress_flag','balance_to_emi_coverage'
]
numeric_features = [f for f in numeric_features if f in feature_table.columns]

corr_with_default = feature_table[numeric_features + ['default_flag']].corr()['default_flag'] \
                                  .drop('default_flag').sort_values()

# Plot top 15 positive and top 15 negative correlations
top_pos = corr_with_default.tail(15)
top_neg = corr_with_default.head(10)
combined = pd.concat([top_neg, top_pos])

fig, ax = plt.subplots(figsize=(10, 10))
colors_corr = [RED if v > 0 else GREEN for v in combined.values]
bars = ax.barh(combined.index, combined.values, color=colors_corr, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Default Flag\n(Red = risk factor, Green = protective factor)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation with default_flag')
plt.tight_layout()
plt.show()

print('\n📌 TOP RISK FEATURES (highest positive correlation with default):')
for feat, val in corr_with_default.tail(8).sort_values(ascending=False).items():
    print(f'   {feat:<35} {val:+.3f}')
print('\n📌 TOP PROTECTIVE FEATURES (negative correlation with default):')
for feat, val in corr_with_default.head(5).items():
    print(f'   {feat:<35} {val:+.3f}')

In [ ]:
# ── Feature correlation heatmap (multicollinearity check) ─
# Important: highly correlated features hurt model interpretability
# We check here so we can drop redundant ones before Phase 3
key_features = [
    'income_to_emi','credit_bureau_score','risk_grade_numeric','stress_score',
    'dpd_max','delinquency_rate','payment_ratio','early_stress_flag',
    'avg_volatility','shock_frequency','cashflow_stress_flag',
    'product_risk_score','channel_risk_score','employment_risk_score'
]
key_features = [f for f in key_features if f in feature_table.columns]

corr_matrix = feature_table[key_features].corr().round(2)

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, vmin=-1, vmax=1, mask=mask,
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix (Multicollinearity Check)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Flag highly correlated pairs
print('\n⚠️  Highly correlated pairs (|r| > 0.7) — consider dropping one in Phase 3:')
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i,j]) > 0.7:
            print(f'   {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: {corr_matrix.iloc[i,j]:.2f}')

## 9 · Save Feature Table

In [ ]:
# ── Save the final feature table ──────────────────────────
feature_table.to_csv(f'{PATH}06_feature_table.csv', index=False)
print(f'✓ Saved: 06_feature_table.csv')
print(f'  Shape : {feature_table.shape[0]:,} rows × {feature_table.shape[1]} cols')

# ── Feature summary for Phase 3 ──────────────────────────
print('\n' + '='*65)
print('  FEATURE ENGINEERING SUMMARY — HAND OFF TO PHASE 3')
print('='*65)

feature_groups = {
    'Affordability (4)':    ['income_to_emi','emi_to_income_pct','total_debt_burden_pct','loan_to_income_ratio'],
    'Credit Quality (5)':   ['credit_bureau_score','credit_score_band','thin_file_flag','credit_history_score','risk_grade_numeric'],
    'Repayment (7)':        ['payment_ratio','dpd_max','dpd_avg','delinquency_rate','missed_emi_rate','partial_payment_rate','dpd_trend_slope'],
    'Early Warning (3)':    ['early_delinquency_flag','early_stress_flag','stress_score'],
    'Behavioral (6)':       ['avg_volatility','spending_shock_count','avg_cashflow_score','balance_stability_ratio','shock_frequency','cashflow_stress_flag'],
    'Product/Pricing (4)':  ['rate_spread','long_tenure_flag','product_risk_score','balance_to_emi_coverage'],
    'Acquisition (3)':      ['channel_risk_score','cac_to_loan_ratio','approval_turnaround_days'],
    'Demographics (3)':     ['age','employment_risk_score','city_tier_risk'],
}

total_features = 0
for group, feats in feature_groups.items():
    available = [f for f in feats if f in feature_table.columns]
    total_features += len(available)
    print(f'\n  {group}')
    for f in available:
        print(f'    ✓ {f}')

print(f'\n  TOTAL FEATURES BUILT: {total_features}')
print(f'  TARGET VARIABLE    : default_flag (12.4% positive rate)')
print('\n  ✅ Phase 2 Complete! Ready for Phase 3: ML Models')
print('='*65)